In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nna
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
from torch.autograd import Variable

#Importing the dataset
movies=pd.read_csv('movies.dat',sep='::',header=None,engine='python',encoding='latin-1')
users=pd.read_csv('users.dat',sep='::',header=None,engine='python',encoding='latin-1')
ratings=pd.read_csv('ratings.dat',sep='::',header=None,engine='python',encoding='latin-1')

#Preparing the training set and the test set
training_set=pd.read_csv('u1.base',delimiter='\t')
training_set=np.array(training_set,dtype='int')
test_set=pd.read_csv('u1.test',delimiter='\t')
test_set=np.array(test_set,dtype='int')




In [4]:
#Getting the number of user and movies
nb_users=int(max(max(training_set[:,0]),max(test_set[:,0])))
nb_movies=int(max(max(training_set[:,1]),max(test_set[:,1])))

In [5]:
#Converting the data into an array with user in lines and movies in columns
def convert(data):
    new_data=[]
    for id_users in range(1,nb_users+1):
        id_movies=data[:,1][data[:,0]==id_users]
        id_ratings=data[:,2][data[:,0]==id_users]
        ratings=np.zeros(nb_movies)
        ratings[id_movies-1]=id_ratings
        new_data.append(list(ratings))
    return new_data
training_set=convert(training_set)
test_set=convert(test_set)

In [6]:
nb_users

943

In [7]:
nb_movies


1682

In [8]:
#Converting the data into torch binary
training_set=torch.FloatTensor(training_set)
test_set=torch.FloatTensor(test_set)

In [9]:
#Converting the ratings into binary ratings 1 (liked) or 0 (Not liked)
training_set[training_set==0]=-1
training_set[training_set==1]=0
training_set[training_set==2]=0
training_set[training_set>=3]=1
test_set[test_set==0]=-1
test_set[test_set==1]=0
test_set[test_set==2]=0
test_set[test_set>=3]=1

#Creating the architecture of the neural Network
class RBM():
    def __init__(self,nv,nh):
        self.W=torch.randn(nh,nv) #100*1682
        self.a=torch.randn(1,nh) #1*100
        self.b=torch.randn(1,nv) #1*1682
    def sample_h(self,x):
        wx=torch.mm(x,self.W.t())
        activation=wx+self.a.expand_as(wx)
        p_h_given_v=torch.sigmoid(activation) #100*100
        return p_h_given_v,torch.bernoulli(p_h_given_v)
    def sample_v(self,y):
        wy=torch.mm(y,self.W) #100*1682
        activation=wy+self.b.expand_as(wy)
        p_v_given_h=torch.sigmoid(activation)
        return p_v_given_h,torch.bernoulli(p_v_given_h)
    def train(self,v0,vk,ph0,phk):
        self.W+=(torch.mm(v0.t(),ph0)-torch.mm(vk.t(),phk)).t()
        #!=1682*100*100*100+1682*100*1682*100
        self.b+=torch.sum((v0-vk),0)
        self.a+=torch.sum((ph0-phk),0)
        

In [10]:
nv=len(training_set[0])
nh=100
batch_size=100
rbm=RBM(nv,nh)
print(rbm.W.size())

torch.Size([100, 1682])


In [11]:
#Training the rBM
nb_epoch=10
for epoch in range(1,nb_epoch+1):
    train_loss=0
    s=0.
    for id_user in range(0, nb_users-batch_size,batch_size):
        vk=training_set[id_user:id_user+batch_size]
        v0=training_set[id_user:id_user+batch_size]
        ph0,_=rbm.sample_h(v0)
        for k in range(10):
            _,hk=rbm.sample_h(vk)
            _,vk=rbm.sample_v(hk)
            vk[v0<0]=v0[v0<0]
        phk,_=rbm.sample_h(vk)
        rbm.train(v0,vk,ph0,phk)
        train_loss+=torch.mean(torch.abs(v0[v0>=0]-vk[v0>=0]))
        s+=1.
    print('epoch:'+str(epoch)+'loss:'+str(train_loss/s))

epoch:1loss:tensor(0.3371)
epoch:2loss:tensor(0.2208)
epoch:3loss:tensor(0.2493)
epoch:4loss:tensor(0.2495)
epoch:5loss:tensor(0.2487)
epoch:6loss:tensor(0.2484)
epoch:7loss:tensor(0.2467)
epoch:8loss:tensor(0.2487)
epoch:9loss:tensor(0.2484)
epoch:10loss:tensor(0.2476)


In [12]:
#Testing the rbm
test_loss=0
s=0.
for id_user in range(nb_users):
    v=training_set[id_user:id_user+1]
    vt=test_set[id_user:id_user+1]
    if len(vt[vt>=0])>0:
        _,h=rbm.sample_h(v)
        _,v=rbm.sample_v(h)
        test_loss+=torch.mean(torch.abs(vt[vt>=0]-v[vt>=0]))
        s+=1
print('test loss: '+str(test_loss/s))
#test loss:tensor(0.2474)

test loss: tensor(0.2407)


In [13]:
#AutoEncoder
#IMportimg the libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
from torch.autograd import Variable

#IMporting the dataset
movies=pd.read_csv('movies.dat',sep='::',header=None,engine='python',encoding='latin-1')
users=pd.read_csv('users.dat',sep='::',header=None,engine='python',encoding='latin-1')
ratings=pd.read_csv('ratings.dat',sep='::',header=None,engine='python',encoding='latin-1')

#Preparing the training set and the test set
training_set=pd.read_csv('u1.base',delimiter='\t')
training_set=np.array(training_set,dtype='int')
test_set=pd.read_csv('u1.test',delimiter='\t')
test_set=np.array(test_set,dtype='int')


In [14]:
#Getting the number of user and movies
nb_users=int(max(max(training_set[:,0]),max(test_set[:,0])))
nb_movies=int(max(max(training_set[:,1]),max(test_set[:,1])))

In [15]:
#Converting the data into an array with user in lines and movies in columns
def convert(data):
    new_data=[]
    for id_users in range(1,nb_users+1):
        id_movies=data[:,1][data[:,0]==id_users]
        id_ratings=data[:,2][data[:,0]==id_users]
        ratings=np.zeros(nb_movies)
        ratings[id_movies-1]=id_ratings
        new_data.append(list(ratings))
    return new_data
training_set=convert(training_set)
test_set=convert(test_set)

In [16]:
training_set=torch.FloatTensor(training_set)
test_set=torch.FloatTensor(test_set)

In [17]:
#creating the architecture of the neural Networks
class SAE(nn.Module):
    def __init__(self,):
        super(SAE,self).__init__()
        self.fc1=nn.Linear(nb_movies,20)
        self.fc2=nn.Linear(20,10)
        self.fc3=nn.Linear(10,20)
        self.fc4=nn.Linear(20,nb_movies)
        self.activation=nn.Sigmoid()
    def forward(self,x):
        x=self.activation(self.fc1(x))
        x=self.activation(self.fc2(x))
        x=self.activation(self.fc3(x))
        x=self.fc4(x)
        return x
sae=SAE()
criterion=nn.MSELoss()
optimizer=optim.RMSprop(sae.parameters(),lr=0.01,weight_decay=0.5)

In [ ]:
#Training the SAE
nb_epoch=200
for epoch in range(1,nb_users+1):
    train_loss=0
    s=0
    for id_user in range(nb_users):
        input=Variable(training_set[id_user]).unsqueeze(0)
        target=input.clone()
        if torch.sum(target.data>0)>0:
            output=sae(input)
            target.require_grad=False
            output[target==0]=0
            loss=criterion(output,target)
            mean_corrector=nb_movies/float(torch.sum(target.data>0)+1e-10)
            loss.backward()
            train_loss+=np.sqrt(loss.data*mean_corrector)
            s+=1
            optimizer.step()
    print('epoch:'+str(epoch)+'loss:'+str(train_loss/s))
        

C:\Users\Brajesh Chaudhary\AppData\Local\Temp\ipykernel_18244\885978042.py:16: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  train_loss+=np.sqrt(loss.data*mean_corrector)


epoch:1loss:tensor(1.7706)
epoch:2loss:tensor(1.0967)
epoch:3loss:tensor(1.0534)
epoch:4loss:tensor(1.0382)
epoch:5loss:tensor(1.0309)
epoch:6loss:tensor(1.0266)
epoch:7loss:tensor(1.0239)
epoch:8loss:tensor(1.0218)
epoch:9loss:tensor(1.0208)
epoch:10loss:tensor(1.0197)
epoch:11loss:tensor(1.0189)
epoch:12loss:tensor(1.0185)
epoch:13loss:tensor(1.0180)
epoch:14loss:tensor(1.0176)
epoch:15loss:tensor(1.0171)
epoch:16loss:tensor(1.0169)
epoch:17loss:tensor(1.0169)
epoch:18loss:tensor(1.0166)
epoch:19loss:tensor(1.0166)
epoch:20loss:tensor(1.0160)
epoch:21loss:tensor(1.0162)
epoch:22loss:tensor(1.0159)
epoch:23loss:tensor(1.0158)
epoch:24loss:tensor(1.0159)


In [2]:
#Testing the SAE
test_loss=0
s=0
for id_user in range(nb_users):
    input=Variable(training_set[id_user]).unsequeeze(0)
    target=Variable(test_set[id_user])
    if torch.sum(target.data>0)>0:
        output=sae(input)
        target.require_grad=False
        output[(target==0).unsqueeze(0)]=0
        loss=criterion(output,target)
        mean_corrector=np.sqrt(loss.data*mean_corrector)
        s=+1.
print('test loss:'+str(test_loss/s))

NameError: name 'nb_users' is not defined

In [ ]:
print('test loss:'+str(test_loss/s))